# RAG(Retrieval-Augmented Generation）检索增强生成
RAG 是目前企业级大模型落地中最核心、最广泛采用的技术架构。


## 文档解析与 Token 级 Dynamic Chunking 算法
1. 理解 RAG 全流程架构：明确离线索引（Indexing）与在线检索生成（Retrieval & Generation）的运作链路。
2. 掌握多样化文档解析（Parsing）：处理纯文本、Markdown、PDF 等格式，应对富文本中的表格与非结构化数据。
3. 深入文本切片（Chunking）机制：
    * Fixed-size Chunking（固定长度切片）：简单但易丢失上下文。
    * Recursive Character Chunking（递归字符切片）：按句号、换行符等自然分界符递归切割。
    * Token-based Dynamic Chunking（基于 Token 的动态切片）：结合 `Tiktoken` 精确控制 Token 数量，避免模型 Context Window 溢出。

## 为什么 Chunking（切片）对 RAG 至关重要？
RAG 的核心逻辑是将外部知识库切割成小块（Chunks），向量化后存入向量数据库。当用户提问时，检索最相关的小块喂给 LLM。

**切片过大（Over-chunking）：**

   * 优点：保留了完整的上下文。
   * 缺点：夹带大量噪音，降低向量检索的精准度（Noise Ratio 高）；占用 LLM 的 Token 窗口，增加 API 成本。

**切片过小（Under-chunking）：**

   * 优点：向量语义聚焦，检索精准。
   * 缺点：丢掉了关键的前后文依赖，LLM 拿到切片后可能因信息缺失而“胡言乱语”。

因此，合理的切片策略（按语义分块、重叠窗口 Overlap、按 Token 精确控制） 是决定 RAG 最终效果的生死线。


**核心切片策略对比**

| 切片策略                                | 原理                                         | 适用场景                    | 优缺点                             |
|-------------------------------------|--------------------------------------------|-------------------------|---------------------------------|
| Fixed Size（固定字符）                    | 每N个字符硬性截断                                  | 快速Demo/格式规整的纯文本         | 实现简单，但极其容易切断句子或单词               |
| Recursive Character(递归字符)           | 依次尝试 `["\n\n", "\n", "。", "！", " "]`       | 绝大多数通用文档                | 生产首选，极大程度保留段落与句子的完整性            |
| Token-based Dynamic(基于 Token 的动态切片) | 结合 Tiktoken 计算 Token 数，动态调整 Sliding Window | 需要精确控制 Prompt Token 的场景 | 规避跨 Tokenizer 计费差异，严格符合 LLM 限制  |
| Semantic Chunking                   | 计算相邻句子的 Embedding 相似度，变化剧烈处断开              | 结构极其不明显的长文章             | 效果极佳，但解析耗时且需要多次调用 Embedding API |


下面的代码展示了如何使用 Python 原生逻辑与 `tiktoken` 库，实现一个支持 `自然分隔符递归切割 + Token 数量上限限制 + 滑动窗口重叠（Overlap）` 的生产级 Chunking 工具。

In [ ]:
import tiktoken

class TokenBasedTextSplitter:
    """
    基于 Token 计数的递归动态切片器 (Token-based Recursive Chunking)
    """
    def __init__(
        self,
        max_tokens: int = 200,
        overlap_tokens: int = 30,
        model_name: str = "gpt-4o"
    ):
        self.max_tokens = max_tokens
        self.overlap_tokens = overlap_tokens
        self.tokenizer = tiktoken.encoding_for_model(model_name)
        # 递归分隔符优先级：段落 -> 换行 -> 中英文句号 -> 空格 -> 字符
        self.separators = ["\n\n", "\n", "。", "！", "？", ". ", " "]

    def count_tokens(self, text: str) -> int:
        """精确计算文本的 Token 数量"""
        return len(self.tokenizer.encode(text))

    def split_text(self, text: str) -> list[str]:
        """对文本进行切片的主逻辑"""
        # 如果整段文本本身就不超限，直接返回
        if self.count_tokens(text) <= self.max_tokens:
            return [text]

        # 1. 尝试寻找最高优先级的分隔符拆分文本
        split_docs = self._split_by_separators(text, self.separators)

        # 2. 将拆分后的片段合并成满足 max_tokens 且带 overlap 的 Chunks
        chunks = self._merge_splits(split_docs)
        return chunks

    def _split_by_separators(self, text: str, separators: list[str]) -> list[str]:
        """按分隔符递归拆分"""
        if not separators:
            return list(text) # 兜底：字符级拆分

        separator = separators[0]
        new_separators = separators[1:]
        splits = text.split(separator)

        final_splits = []
        for s in splits:
            if not s:
                continue
            if self.count_tokens(s) <= self.max_tokens:
                final_splits.append(s)
            else:
                # 递归使用更细粒度的分隔符
                final_splits.extend(self._split_by_separators(s, new_separators))
        return final_splits

    def _merge_splits(self, splits: list[str]) -> list[str]:
        """合并零散片段，加入 Overlap 保持上下文连贯"""
        chunks = []
        current_chunk = []
        current_tokens = 0

        for split in splits:
            split_tokens = self.count_tokens(split)

            if current_tokens + split_tokens > self.max_tokens:
                if current_chunk:
                    chunk_text = "".join(current_chunk)
                    chunks.append(chunk_text)

                    # 构建重叠 (Overlap)：保留末尾部分 token
                    overlap_chunk = []
                    overlap_len = 0
                    for item in reversed(current_chunk):
                        item_tokens = self.count_tokens(item)
                        if overlap_len + item_tokens <= self.overlap_tokens:
                            overlap_chunk.insert(0, item)
                            overlap_len += item_tokens
                        else:
                            break
                    current_chunk = overlap_chunk
                    current_tokens = overlap_len

            current_chunk.append(split)
            current_tokens += split_tokens

        if current_chunk:
            chunks.append("".join(current_chunk))

        return chunks


# -------------------------------------------------------------
# 测试运行
# -------------------------------------------------------------
if __name__ == "__main__":
    sample_document = """
检索增强生成（Retrieval-Augmented Generation，简称 RAG）是一种通过从外部知识库中检索相关信息来增强大语言模型（LLM）输出质量的技术。

RAG 系统通常分为两个核心阶段：离线索引阶段和在线检索生成阶段。
在离线索引阶段，系统首先对原始文档进行解析与清洗，将长文档按逻辑切割成多个连续的文本块（Chunks）。随后，使用 Embedding 模型将这些文本块转换为高维向量，并存入向量数据库中。

在在线检索阶段，当用户提出问题时，系统会计算问题向量与数据库中各切片向量的相似度，召回 Top-K 个最相关的切片。最终，这些切片作为 Prompt 的上下文背景与用户问题一同输入给 LLM，由 LLM 生成精准且具备事实依据的回答。

良好的文本切片（Chunking）策略是 RAG 系统的关键工程细节之一。切片过大会引入冗余噪音，切片过小则会导致语义碎片化。
    """

    # 初始化切片器：每块最多 100 个 Token，重叠 20 个 Token
    splitter = TokenBasedTextSplitter(max_tokens=100, overlap_tokens=20)
    chunks = splitter.split_text(sample_document)

    print(f"📄 原始文档已成功切割为 {len(chunks)} 个小块 (Chunks):\n" + "="*50)
    for idx, chunk in enumerate(chunks, 1):
        token_count = splitter.count_tokens(chunk)
        print(f"🔹 [Chunk {idx}] (Token数: {token_count}):")
        print(f"\"{chunk.strip()}\"\n" + "-"*40)


1. PDF/Markdown 富文本中的表格处理：
    * 在处理带有 HTML/Markdown 表格（`<table>` 或 `| header | header` |）的文档时，如果直接用简单的字符拆分，表格往往会被切得七零八落，导致 LLM 完全失去行列对齐的阅读能力。在生产环境中，你会采用什么策略来保障表格数据的完整性？
        * HTML 表格：使用 HTML 解析器（如 Python 的 BeautifulSoup 或 lxml）构建 DOM 树
        * Markdown 解析器（如 `mistune` 开启 `tables` 插件，或 `markdown-it-py`）来识别表格边界，而不是逐行扫描。解析后同样转为二维数组或 `DataFrame`。
        * PDF/图片中的表格：需要 OCR + 布局分析。Camelot、Tabula 适合带线表格的 PDF；PaddleOCR + 自定义后处理可以处理扫描件。关键是识别出单元格边界后，再按位置关系重建行列。
        * 原始文档 (HTML/Markdown/PDF) -> 专用解析器 (DOM/Markdown parser/OCR) -> 结构化中间表示 (DataFrame / JSON) -> 结构校验 + 合并单元格展开 + 类型推断 -> 元数据增强 (列名、类型、描述) -> 按场景选择送入方式：
        - 小表 → 整表 JSON 入 prompt
        - 大表 → 结构预览 + 行级检索
        - RAG  → 行级向量索引 + 混合检索

2. 切片重叠（Overlap）的副作用：
    * 设置适当的 Overlap 可以防止信息在断句处丢失，但如果 Overlap 设置得太大（例如达到 Chunk 长度的 50%），在后续向量检索 Top-K 结果时可能会引发什么问题？对向量数据库和 LLM 提示词有什么隐患？
        * Overlap 设为 Chunk 长度的 50% 会引发全链路连锁反应：向量库端导致索引膨胀与检索重复泛滥；LLM 端导致上下文浪费、注意力稀释及引用回声。
        * 对向量数据库的隐患:
            * 索引膨胀与成本飙升
            * 检索结果重复泛滥
            * Rerank重排 算力严重浪费
            * 缓存效率大幅下降
        * 对 LLM 提示词的隐患
            * 上下文窗口被重复文本挤占
            * 注意力稀释与忽略中间关键信息
            * 引用回声与答案质量下降
            * 评估漂移与调试困难

## RAG 系统的核心数据桥梁
文本向量化（Embedding）、向量空间相似度度量，以及向量数据库（Vector Database）的工程落地

1. 理解 Embedding 的本质：掌握文本如何映射为高维密集向量（Dense Vector），以及为什么传统 Sparse Vector（如 TF-IDF, BM25）与 Dense Vector 互补。
2. 掌握三大向量相似度度量指标：余弦相似度（Cosine Similarity）、欧氏距离（L2 Distance / Euclidean Distance）、点积（Dot Product / Inner Product）。
3. 掌握主流向量数据库实战：学习轻量级/嵌入式向量库（Chroma / LanceDB）与企业级分布式向量库（Milvus）的架构与选型。


#### Embedding 与向量空间

**什么是 Embedding？**

在 RAG 系统中，文本切片（Chunk）是人类可读的字符串，大模型无法直接对其进行数学计算。Embedding 模型（如 `text-embedding-3-small`、`bge-large-zh`）通过深度神经网络，将任意长度的文本转换成固定维度的浮点数数组（例如 `1536` 维或 `1024` 维） 。

   * 语义对齐：语义相近的文本，在向量空间中的距离越近（如“苹果公司”和“iPhone”）；语义无关的文本，在向量空间中互相正交或距离较远。

**三大相似度度量方式对比**

在向量数据库检索 Top-K 相关片段时，计算两个向量 $\vec{u}$ 和 $\vec{v}$ 的相似度有三种最常用的数学公式：

|相似度指标|数学公式|特性与适用场景|关键注意事项|
|------|------|------|------|
|余弦相似度 (Cosine Similarity)|$\cos(\theta) = \frac{\vec{u} \cdot \vec{v}}{\Vert{}\vec{u}\Vert{} \Vert{}\vec{v}\Vert{}}$|衡量向量间的方向夹角，取值范围 $[-1, 1]$。广泛应用于文本语义检索，不敏感于文本长度带来的模长变化。|评估语义匹配的首选。|
|点积 (Dot Product / IP)|$\text{IP} = \vec{u} \cdot \vec{v}$|计算向量的点积。计算速度极快。|仅当向量已经过归一化 (L2 Normalized, $\Vert{}\vec{v}\Vert{}=1$) 时，点积才等价于余弦相似度。|
|欧氏距离 (Euclidean / L2)|$d = \sqrt{\sum (u_i - v_i)^2}$|衡量高维空间中的绝对距离。数值越小表示越相似（距离越近）。|对向量模长较敏感，通常用于图片特征向量或不需要归一化的场景。|


#### 向量数据库的核心原理：HNSW 索引
如果数据库中有 100 万个文本切片，每次查询都与 100 万个向量做全量点积计算（暴力检索 / Flat），延迟将极高。

生产级向量数据库（如 Milvus, Chroma, Pinecone） 普遍采用 ANN（Approximate Nearest Neighbor，近似最近邻算法） 索引，其中最著名的算法是 HNSW（Hierarchical Navigable Small World，分层可导航小世界网格）：

   * 原理：建立类似于跳表（Skip-list）的多层图结构。上层为稀疏图，用于快速粗粒度定位；下层为密集图，用于细粒度局部搜索。
   * 效果：将时间复杂度从 $O(N)$ 降至 $O(\log N)$，毫秒级返回 Top-K 结果。


下面的代码演示了如何使用 Python 调用 OpenAI/HuggingFace Embedding 模型，将切片数据存入 ChromaDB，并进行相似度检索：

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. 初始化本地持久化 Chroma 向量数据库客户端
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 2. 配置 Embedding 函数 (以 OpenAI text-embedding-3-small 为例)
# 如果本地运行，也可以换成 SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-zh-v1.5")
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY", "your-api-key"),
    model_name="text-embedding-3-small"
)

# 3. 创建或获取集合 (Collection)
# 指定 distance_metric 为 cosine (余弦相似度)
collection = chroma_client.get_or_create_collection(
    name="rag_knowledge_base",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"}
)

# 4. 模拟上一天 Chunking 算法输出的文档切片数据
documents = [
    "检索增强生成（RAG）结合了外部知识库检索与大模型生成，能有效缓解大模型幻觉问题。",
    "向量数据库（Vector Database）如 Milvus 和 Chroma，专门用于高效存储与检索高维 Embedding 向量。",
    "Token-based Chunking 策略利用 Tiktoken 精确控制文本分块的 Token 数量，避免模型上下文溢出。",
    "Transformer 架构的核心是 Self-Attention 机制，它允许模型在序列中关注不同位置的依赖关系。"
]

metadatas = [
    {"source": "rag_intro.pdf", "chapter": 1},
    {"source": "vector_db_guide.md", "chapter": 2},
    {"source": "chunking_strategies.txt", "chapter": 1},
    {"source": "transformer_paper.pdf", "chapter": 1}
]

ids = [f"doc_chunk_{i}" for i in range(len(documents))]

# 5. 写入数据到向量数据库
print("📥 正在将切片向量化并写入向量数据库...")
collection.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)
print("✅ 写入完成！")

# 6. 运行相似度检索测试
query_text = "如何避免 RAG 切片超出 LLM 的 Token 限制？"
print(f"\n🔍 用户查询: \"{query_text}\"")

results = collection.query(
    query_texts=[query_text],
    n_results=2 # 召回 Top-2 最相关的切片
)

print("\n🎯 检索结果 (Top-K Chunks):")
for idx, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0]), 1):
    # Chroma 在 cosine 距离下返回的是 Distance = 1 - CosineSimilarity
    similarity = 1 - dist
    print(f"\n[结果 {idx}] (相似度: {similarity:.4f})")
    print(f"  内容: {doc}")
    print(f"  元数据: {meta}")


1. 密集向量（Dense Vector）与稀疏向量（Sparse Vector / BM25）的局限：
    * 假设用户检索产品型号 `"Model-X9000-Pro"` 或特定专业术语，使用纯 Embedding 向量检索经常会出现查不到或匹配错的情况（因为 Embedding 偏向语义概括，对精确字符匹配不敏感）。在工业级 RAG 中，混合检索（Hybrid Search: Dense + Sparse） 是如何解决这个问题的？

2. 向量数据库的元数据过滤（Metadata Filtering）：
    * 如果你需要在一个拥有 1000 万篇文档的向量库中检索仅属于 `user_id = 123` 的私有文档，是应该先向量检索再过滤元数据（Post-filtering），还是先元数据过滤再向量检索（Pre-filtering）？为什么？
        * 先元数据过滤再向量检索.
        * 核心原因在于：私有文档过滤属于高选择性、硬性业务约束场景，Pre-filtering 能在保障安全隔离的前提下实现最优性能与召回质量。

#### 混合检索（Hybrid Search: BM25 + Dense Vector）与重排机制（Reranking, Cross-Encoder）
文本向量化（Embedding）、三大相似度指标以及向量数据库（Vector DB)仅仅能让项目跑通，然而在实际生产环境中，纯向量检索（Dense Retrieval）往往无法单独满足高准确率的要求。

1. 痛点剖析：理解纯向量检索的局限性（如专有名词、产品型号、缩写检索失效问题）。
2. 掌握混合检索（Hybrid Search）：掌握稀疏检索（Sparse / BM25）与密集检索（Dense Vector）互补原理，理解 RRF（Reciprocal Rank Fusion） 融合打分算法。
3. 彻底搞懂 Bi-Encoder 与 Cross-Encoder：理解为什么向量检索模型（Bi-Encoder）适合粗召回，而重排模型（Cross-Encoder / Reranker）适合精排序。
4. 代码实战：使用 Python 手写完整的“BM25 + 向量检索 + Cross-Encoder 重排”两阶段检索 Pipeline。


#### 混合检索与 Reranking
1. 为什么纯向量检索不够用？

    Embedding 模型（Dense Vector）将文本映射到连续的语义空间中，擅长捕获泛化的语义相似度，但在以下场景容易失真：
   * 精确匹配失效：检索具体的订单号（如 `ORD-2026-8891`）、型号（如 `RTX-4090-Ti`）或代码函数名（如 `get_user_by_id()`）时，Embedding 倾向于将它们视作无差别或相似的 Token。
   * 生僻词与专有名词：领域内特有缩写（如医疗、法律条款）在通用 Embedding 模型的词表中权重可能较低。
   * 解决方案：引入基于词频和关键字倒排索引的传统检索算法 BM25（Sparse Vector），与向量检索进行双路召回（Hybrid Search）。

2. 双路召回融合算法：RRF (Reciprocal Rank Fusion)
    当我们同时使用 BM25 和 向量检索 分别召回了 Top-N 个结果后，由于 BM25 输出的是非标准化得分（可能是 `0~50`），而余弦相似度输出的是 `0~1`，两种得分无法直接相加。
   企业级常用的无参融合算法是 RRF（倒数排名融合）：
   $$\text{RRF\_Score}(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$

    * $M$：检索系统集合（即 `[BM25, Dense]`）。
    * $r_m(d)$：文档 $d$ 在系统 $m$ 中的排名位置（从 1 开始）。
    * $k$：常数平滑因子（通常取 $k = 60$）。

    > RRF 的优势：只依赖相对排名（Rank），不依赖原始分数，非常鲁棒且无需训练参数。

3. 两阶段检索架构：Bi-Encoder vs Cross-Encoder

    为了在高并发毫秒级响应与深度语义理解之间取得平衡，生产级 RAG 普遍采用“粗召回 + 精重排”的两阶段架构：

   | 维度    | Bi-Encoder（双编码器 / Embedding 模型）                       | Cross-Encoder（交叉编码器 / Reranker）                                |
   |-------|-------------------------------------------------------|----------------------------------------------------------------|
   | 计算架构  | Query 和 Document 分别输入 模型提取向量，计算余弦距离。                  | Query 和 Document 拼接在一起 (`[CLS] Query [SEP] Doc [SEP]`) 共同输入模型。 |
   | 注意力交互 | Query 与 Doc 在 Transformer 内部无注意力交互（Late Interaction）。 | Query 与 Doc 的每个 Token 之间做全量 Self-Attention 交叉关注。               |
   | 计算效率  | 极高（文档向量可离线建立 HNSW 索引，查询时仅做线性代数计算）。                    | 较低（每篇文档都需要实时经过 Transformer 层前向传播计算）。                           |
   | 匹配精度  | 适合在大规模库中快速缩小范围（如 1,000,000 找 Top 50）                  | 适合对少数文档做精细打分与排序（如 Top 50 选 Top 3）。                             |


构建“BM25 + 向量 + Reranker”两阶段检索系统,下面使用 Python 模拟完整的混合检索与 Cross-Encoder 重排流程：

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# 1. 准备测试知识库数据
documents = [
    "大模型 RAG 系统中，混合检索（Hybrid Search）结合了 BM25 关键字匹配与 Dense 向量检索的优点。",
    "BM25 算法依赖词频（TF）和逆文档频率（IDF），对精确的产品型号、订单编号和专有名词检索效果极佳。",
    "Transformer 架构利用 Self-Attention 机制捕获长距离上下文依赖关系。",
    "Reranker（重排模型）采用 Cross-Encoder 架构，将 Query 和 Document 拼接输入，打分精度远高于 Bi-Encoder。",
    "向量数据库通过 HNSW 等近似最近邻算法实现毫秒级高维向量相似度搜索。"
]

# 分词处理（此处用简单按字/词切分模拟，生产环境中文推荐 jieba）
tokenized_corpus = [doc.replace("，", " ").replace("。", " ").split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

query = "为什么 RAG 系统需要使用 BM25 和 Reranker 重排？"
tokenized_query = query.replace("？", "").split()

# ----------------------------------------------------
# 步骤 A: BM25 稀疏检索召回
# ----------------------------------------------------
bm25_scores = bm25.get_scores(tokenized_query)
bm25_top_indices = np.argsort(bm25_scores)[::-1]  # 降序排列

print("🔍 --- BM25 检索得分排名 ---")
for idx in bm25_top_indices:
    print(f"得分: {bm25_scores[idx]:.4f} | 文档: {documents[idx]}")

# ----------------------------------------------------
# 步骤 B: Cross-Encoder (Reranker) 深度精重排
# ----------------------------------------------------
# 使用轻量级开源 Cross-Encoder 模型 (如 BAAI/bge-reranker-base 或 cross-encoder/ms-marco-MiniLM-L-6-v2)
print("\n🎯 --- 正在加载 Cross-Encoder 重排模型进行精排序 ---")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 构建 (Query, Document) 对
pairs = [[query, doc] for doc in documents]
rerank_scores = reranker.predict(pairs)

# 根据 Reranker 得分重新排序
reranked_indices = np.argsort(rerank_scores)[::-1]

print("\n🚀 --- Reranker 精排最终输出 (Top-K) ---")
for rank, idx in enumerate(reranked_indices[:3], 1):
    print(f"[Top {rank}] 重排得分: {rerank_scores[idx]:.4f}")
    print(f"       内容: {documents[idx]}\n")

1. 性能与延迟权衡：
    * 在千万人级别的大规模知识库中，如果直接对召回的 Top 500 篇文档全部用 Cross-Encoder 进行 Rerank，会导致严重的接口超时。工程上应该如何控制 Rerank 的召回数量（Top-K Cutoff）？

2. 混合检索权重配置
    * 除了 RRF 之外，某些数据库支持显式设定 `Alpha * Dense_Score + (1 - Alpha) * Sparse_Score`。如果你的 RAG 系统主要用于医疗处方与药品名称查询，`Alpha` 应该偏向 0（偏向 Sparse）还是偏向 1（偏向 Dense）？为什么？
